In [ ]:
import os

from  dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from IPython.display import Image, display

load_dotenv(".env")

model = init_chat_model(
    model="deepseek-v4-pro",  
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# Tool Calls
- Agent 支持静态和动态绑定工具，后者需要用到中间件
- 内置工具：https://docs.langchain.com/oss/python/integrations/tools

## 1. Custom Tools

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from rich import print as rprint


class WeatherInput(BaseModel):
    city: str = Field(description="城市名称，例如北京、上海、广州")
    include_forecast: bool = Field(
        default=False,
        description="是否查询未来几天的天气预报。如果用户问'这几天'、'未来几天'，则为 True"
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, include_forecast: bool = False) -> dict:
    """获取指定城市的天气信息。"""
    result = {
        "city": city,
        "temperature": "25°C",
        "condition": "晴天",
    }

    if include_forecast:
        result["forecast"] = [
            {"day": "Monday", "temperature": "26°C", "condition": "多云"},
            {"day": "Tuesday", "temperature": "24°C", "condition": "小雨"},
            {"day": "Wednesday", "temperature": "27°C", "condition": "晴天"},
        ]

    return result


agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="你是一个天气查询助手，根据用户的提问查询天气信息。如果问题与天气无关，请礼貌地告知用户。",
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="北京这几天的天气怎么样？")
    ]
})
display(Image(agent.get_graph().draw_mermaid_png())) # 可视化
rprint(response)

## 2. Built-in Tools

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(".env")


class SearchInput(BaseModel):
    query: str = Field(..., description="需要搜索验证的问题或关键词")


class NobelWinner(BaseModel):
    name: str = Field(..., description="得主姓名或组织名称")
    category: str = Field(..., description="诺贝尔奖类别，例如物理学、化学、生理学或医学、文学、和平奖、经济学")
    nationality: str = Field(..., description="国籍；如果是组织则写其所属国家/地区或'组织'")
    achievements: list[str] = Field(..., description="获奖原因或相关成就")


class NobelAnswer(BaseModel):
    year: int = Field(..., description="年份")
    winners: list[NobelWinner] = Field(..., description="该年度诺贝尔奖得主列表")
    sources: list[str] = Field(..., description="搜索验证所依据的来源链接")


tavily = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=5,
    search_depth="basic",
    topic="general",
    include_answer=True,
)

model_with_tools = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)


@tool(args_schema=SearchInput)
def web_search(query: str) -> str:
    """联网搜索并返回用于事实核验的结果。"""
    result = tavily.invoke({"query": query})
    return json.dumps(result, ensure_ascii=False)


system_prompt = """
你是一个专业智能助手。
回答任何涉及事实、时效或名单的问题前，必须先调用 web_search 工具进行搜索验证。
回答时只使用搜索结果中能验证的信息。
如果搜索结果不足以确认，请明确说明无法确认。
"""

agent = create_agent(
    model=model_with_tools,
    tools=[web_search],
    system_prompt=system_prompt,
    response_format=ToolStrategy(NobelAnswer),
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="2025年诺贝尔奖得主有哪些？请按奖项类别列出。")
    ]
})

rprint(response["structured_response"])

## 3. Multiple Tools

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from IPython.display import Image, display
from rich import print as rprint
import random


# 工具 1：天气查询
class WeatherInput(BaseModel):
    city: str = Field(description="城市名称，例如北京、上海、东京、纽约")


def _random_temp(base_min: int, base_max: int) -> str:
    """生成随机温度，范围基于城市常年气温。"""
    return f"{random.randint(base_min, base_max)}°C"


@tool(args_schema=WeatherInput)
def get_weather(city: str) -> dict:
    """查询指定城市的实时天气信息（模拟数据，温度为随机生成）。"""
    weather_db = {
        "北京": {"temperature": _random_temp(20, 38), "condition": "晴", "humidity": "45%"},
        "上海": {"temperature": _random_temp(25, 40), "condition": "多云", "humidity": "70%"},
        "东京": {"temperature": _random_temp(18, 35), "condition": "小雨", "humidity": "80%"},
        "纽约": {"temperature": _random_temp(15, 35), "condition": "晴", "humidity": "50%"},
        "伦敦": {"temperature": _random_temp(10, 28), "condition": "阴", "humidity": "65%"},
        "新加坡": {"temperature": _random_temp(26, 35), "condition": "雷阵雨", "humidity": "85%"},
    }
    return weather_db.get(city, {"temperature": "N/A", "condition": "暂无数据", "humidity": "N/A"})


# 工具 2：汇率查询
class ExchangeInput(BaseModel):
    from_currency: str = Field(description="源货币代码（三位大写），例如 USD、CNY、JPY、EUR")
    to_currency: str = Field(description="目标货币代码（三位大写），例如 USD、CNY、JPY、EUR")


@tool(args_schema=ExchangeInput)
def get_exchange_rate(from_currency: str, to_currency: str) -> dict:
    """查询两种货币之间的实时汇率（模拟数据）。"""
    rates = {
        ("USD", "CNY"): 7.25,
        ("USD", "JPY"): 150.50,
        ("USD", "EUR"): 0.92,
        ("CNY", "USD"): 0.138,
        ("CNY", "JPY"): 20.76,
        ("CNY", "EUR"): 0.127,
        ("JPY", "CNY"): 0.048,
        ("JPY", "USD"): 0.0066,
        ("EUR", "USD"): 1.09,
        ("EUR", "CNY"): 7.88,
    }
    key = (from_currency.upper(), to_currency.upper())
    if key in rates:
        return {"from": from_currency.upper(), "to": to_currency.upper(), "rate": rates[key]}
    return {"error": f"暂不支持 {from_currency} → {to_currency} 的汇率查询"}


# 工具 3：数学计算
class CalcInput(BaseModel):
    expression: str = Field(description="数学表达式，例如 '1000 * 7.25'、'(100 + 200) / 3'")


@tool(args_schema=CalcInput)
def calculate(expression: str) -> dict:
    """执行四则运算和常用数学函数（sqrt, log, sin, cos 等）。"""
    import math
    safe_names = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
    try:
        result = eval(expression, {"__builtins__": {}}, safe_names)
        return {"expression": expression, "result": round(result, 4)}
    except Exception as e:
        return {"expression": expression, "error": str(e)}


# 创建 Agent（一次性绑定三个自定义工具）
agent = create_agent(
    model=model,
    tools=[get_weather, get_exchange_rate, calculate],
    system_prompt=(
        "你是一个多功能智能助手，可以查询天气、汇率，并进行数学计算。\n"
        "规则：\n"
        "1. 遇到需要多个工具配合的任务（如'查汇率后算钱'），依次调用相关工具\n"
        "2. 遇到独立的子任务（如'查A城市天气 + 查B货币汇率'），可以并行调用\n"
        "3. 回答时用清晰的中文整合所有工具返回的结果"
    ),
)

# 可视化 Agent 的工具调用图
display(Image(agent.get_graph().draw_mermaid_png()))

In [ ]:
# 场景 1：并行调用多个工具（两个独立查询，互不依赖）
print("【场景 1】同时查询天气和汇率（互不依赖 → Agent 可并行调用）")
print("-" * 60)

response1 = agent.invoke({
    "messages": [
        HumanMessage(content="帮我查一下东京的天气，以及现在美元兑日元的汇率是多少？")
    ]
})
rprint(response1)

In [ ]:
# 场景 2：串行调用多个工具（汇率 → 计算，后者依赖前者输出）
print("\n\n【场景 2】先查汇率再算金额（后续工具依赖前一个工具的输出 → Agent 串行调用）")
print("-" * 60)

response2 = agent.invoke({
    "messages": [
        HumanMessage(content="我有2000美元，想换成人民币，能换多少？顺便告诉我上海天气如何。")
    ]
})
rprint(response2)

## 4. 如何让大模型准确选对工具

**原理简述**

Agent 本身不"选择"工具 — 它将工具的名称、描述和参数 schema 发给 LLM，由 LLM 根据用户问题做语义匹配，决定是否调用、调用哪个工具。LLM 返回 tool_call 指令（包含工具名和参数），Agent 框架负责执行。

**影响因素优先级**（从高到低）：

工具描述 > 工具名称 > 参数描述 > tool_choice > 工具数量 > System Prompt 引导


**注意**：前三个（描述、名称、参数）是让模型"自己选对"；tool_choice 是外部强制手段，排在它们之后。

### 4.1 工具描述

模型根据描述文字判断是否调用工具。核心原则：**描述「何时调用」，而非仅描述「做什么」**。

一个高质量的工具描述应包含以下字段：

| 字段 | 作用 | 示例 |
|---|---|---|
| **功能概述** | 一句话说清工具做什么 | "查询指定城市的实时天气信息" |
| **触发条件** | 什么情况下应调用，附典型用户问法示例（最关键） | "当用户问天气时调用，如'北京天气'、'要不要带伞'" |
| **边界说明** | 什么情况下**不应**调用 | "不适用于气候特点、空气质量、历史天气" |
| **输出格式** | 返回值结构，帮助模型正确解读结果 | "返回：{城市} 今天{天气}，{温度}度" |

**说明**：原先"触发条件"和"典型问法"是两个独立字段，但本质上"典型问法"就是触发条件的具体实例——把用户原话直接放在触发条件里，比分开写更直观，也减少冗余。

此外，**工具之间要互斥**，多个工具描述相似时模型容易选错。每个工具的**触发条件**和**边界说明**要写出区别于其他工具的关键信息

In [ ]:
@tool
def get_weather(city: str) -> str:
    """[功能] 查询指定城市的实时天气信息。

    [触发条件（含典型问法）]
    当用户询问以下内容时调用此工具：
    - 某城市的天气（"北京天气"、"上海今天多少度"）
    - 是否会下雨/下雪（"明天要带伞吗"）

    [边界]
    以下情况不应调用此工具：
    - 询问气候特点（"北京冬天冷吗"）→ 常识问题
    - 询问空气质量 → 需用其他工具

    [输出]
    返回包含城市名、天气状况和温度的字符串。
    """

In [ ]:
# 4.1 最佳实践：优秀的工具描述
# 核心原则：描述「何时调用」而非仅描述「做什么」
# 四个必要字段：[功能] [触发条件] [边界] [输出]

from typing import Literal

from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from rich import print as rprint


# ── 示例 1：天气查询 ──
class WeatherInput(BaseModel):
    city: str = Field(
        description="城市名称，如'北京'、'上海'、'Tokyo'。不支持经纬度或邮编。"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="温度单位。默认 celsius。"
    )


@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius") -> dict:
    """查询指定城市的实时天气信息。

    当用户询问以下内容时调用此工具：
    - 某城市的天气（"北京天气"、"上海今天多少度"、"东京热不热"）
    - 是否会下雨/下雪/台风（"明天要带伞吗"、"会下雪吗"）
    - 出行穿衣建议相关（"去北京穿什么"）→ 先查天气再给建议

    以下情况不应调用此工具：
    - 询问气候特点（"北京冬天冷吗"）→ 常识问题，直接回答
    - 询问空气质量、紫外线 → 需用其他专用工具
    - 询问历史天气（"上周下雨了吗"）→ 本工具仅支持实时天气
    - 用户只是在聊天气话题而非查询（"今天天气真好"）

    返回 dict：{city, temperature, condition, humidity}
    """
    weather_db = {
        "北京": {"temperature": "32°C", "condition": "晴", "humidity": "45%"},
        "上海": {"temperature": "30°C", "condition": "多云", "humidity": "70%"},
        "东京": {"temperature": "28°C", "condition": "小雨", "humidity": "80%"},
    }
    return weather_db.get(city, {"temperature": "N/A", "condition": "暂无数据"})


# ── 示例 2：空气质量（与天气工具互斥） ──
class AirQualityInput(BaseModel):
    city: str = Field(description="城市名称，如'北京'、'上海'。不支持经纬度或邮编。")


@tool(args_schema=AirQualityInput)
def get_air_quality(city: str) -> dict:
    """查询指定城市的实时空气质量指数（AQI）。

    当用户询问以下内容时调用此工具：
    - 空气质量 / AQI / PM2.5（"北京空气质量"、"今天雾霾吗"、"PM2.5多少"）
    - 是否适合户外活动（"今天适合跑步吗"→ 需结合空气质量判断）

    以下情况不应调用此工具：
    - 询问天气（温度、晴雨、是否带伞）→ 用 get_weather
    - 询问气候趋势 → 常识问题

    返回 dict：{city, aqi, level, pm25, advice}
    """
    return {
        "city": city,
        "aqi": 85,
        "level": "良",
        "pm25": 35,
        "advice": "正常户外活动",
    }

agent = create_agent(
    model=model,
    tools=[get_weather, get_air_quality],
    system_prompt="你是一个助手，按需调用工具。",
)

resp1 = agent.invoke({"messages": [HumanMessage(content="深圳的冬天下雪么？")]})
rprint(resp1)
resp2 = agent.invoke({"messages": [HumanMessage(content="北京现在能穿短袖么？")]})
rprint(resp2)
resp3 = agent.invoke({"messages": [HumanMessage(content="北京最近的空气质量咋样？")]})
rprint(resp3)

### 4.2 工具名称


动词_名词 模式，足够具体以区分同类工具：

- get_weather、search_knowledge_base、calculate_tax
- weather（太泛）、query（查什么？）、lookup（查什么？）

### 4.3 参数描述

每个参数通过 Field(description=...) 写明含义 + 示例；固定可选值用 Literal 约束；可选参数设 default。

In [ ]:
class WeatherInput(BaseModel):
    city: str = Field(
        description="城市名称，如'北京'、'上海'、'Tokyo'。不支持经纬度或邮编。"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", description="温度单位。默认 celsius。"
    )

### 4.4 tool_choice

前面3个要素都是让模型"自己选对"；tool_choice 是**从外部强制指定**调用行为，属于兜底手段，所以排在它们之后。

tool_choice 控制模型是否必须调用工具、调用哪个工具，有四种模式：

| 设置 | 含义 | 适用场景 |
|---|---|---|
| auto | 模型自主决定是否调用（**默认值**） | 正常对话，让模型自己判断 |
| any | 必须调用至少一个工具 | 你确定当前意图需要工具，但不确定具体哪个 |
| {"type": "tool", "name": "xxx"} | 强制调用指定工具 | 上游已做好意图分类，精确路由到某个工具 |
| none | 禁止调用任何工具 | 闲聊 / 不需要工具的纯文本回答 |

**LangChain 用法**：

```python
# bind_tools 时设置
llm_with_tools = llm.bind_tools(tools, tool_choice="any")
```

**使用建议** tool_choice 适合**确定性路由场景**（如意图识别后精确分配工具）。但如果工具定义本身质量差，靠 tool_choice="any" 硬逼模型调用工具，它可能填错参数或选错工具。正确顺序是：**先用前五个要素打磨工具定义，让模型自己选对；再用 tool_choice 兜底确定性场景。**

### 4.5 工具数量

工具越多越易混淆。相似功能合并为一个工具，用参数区分行为。总数控制在 5-10 个以内。

### 4.6 System Prompt

System Prompt 是影响工具选择的**最后一道防线**。在工具定义（描述、名称、参数）都打磨好之后，System Prompt 负责设定**全局行为规则**——它不教模型"用哪个工具"，而是教模型"什么时候必须用工具、什么时候不该用、以及如何整合结果"。

**好 System Prompt 的 5 条核心规则：**

| 规则 | 作用 | 示例 |
|---|---|---|
| **1. 强制调用声明** | 防止模型凭记忆编造 | "回答任何事实性问题前，必须先调用工具验证" |
| **2. 禁止编造数据** | 防止幻觉 | "工具未返回的信息，请明确说'未查到'，不要猜测" |
| **3. 工具调度策略** | 引导并行/串行 | "独立查询可并行调用，有依赖关系的串行调用" |
| **4. 异常处理规则** | 工具失败时的行为 | "如果工具返回错误，告知用户并建议重试，不要假装成功" |
| **5. 回答整合方式** | 输出风格控制 | "用自然语言整合工具结果，保留关键数值，注明数据来源" |

**注意事项：**
- System Prompt 是**软约束**——模型可能不遵守（尤其是弱模型），需要结合 tool_choice 和 middleware 做硬兜底
- 规则要**简短具体**，不要写成"论文"。每条规则一行，模型更容易遵守
- **反面规则比正面规则更有效**——"不要编造数据"比"请如实回答"更管用

In [ ]:
# 4.6 最佳实践：System Prompt 对比实验
@tool
def get_product_stock(product_name: str) -> dict:
    """查询指定商品的库存信息。"""
    stock_db = {
        "iphone 16": {"stock": 128, "price": "¥6,999", "warehouse": "北京仓"},
        "rtx 5090": {"stock": 0, "price": "¥14,999", "warehouse": "暂无"},
        "茅台": {"stock": 56, "price": "¥2,899", "warehouse": "上海仓"},
    }
    return stock_db.get(
        product_name.lower(),
        {"stock": "N/A", "price": "N/A", "warehouse": "N/A", "hint": "未查询到该商品"}
    )

bad_prompt = "你是一个客服助手，可以查询商品信息。"
good_prompt = """
你是一个严谨的客服助手。
规则：
1. 回答任何涉及商品库存、价格的问题前，必须先调用 get_product_stock 验证
2. 工具未返回的信息，必须说"抱歉，我无法回答"，不得编造或猜测
3. 如果用户问的问题和库存/价格无关，直接告知"这不是我的服务范围"
4. 回答时注明数据来源（如"根据库存系统查询"）
5. 工具调用失败时，告知用户"系统暂时不可用，请稍后重试"，不要假装成功
"""

### 4.7 常见问题速查

| 现象 | 根因 | 解决 |
|---|---|---|
| 模型不调工具 | 描述未写触发条件 | docstring 中加"当用户询问...时调用此工具" |
| 模型调错工具 | 多工具描述相似 | 加互斥边界，明确"不适用于..." |
| 参数填错 | 参数描述太抽象 | 加具体示例（如城市名、股票代码） |
| 模型放弃调用 | 必填参数无默认值 | 给可选参数设 `default` |
| 模型凭记忆回答 | System prompt 未要求查工具 | 加"必须先调用工具，不得编造数据" |

## 5. How to Know When an Agent Finishes

**问题**：Agent 在一次 `invoke` 中可能进行多轮工具调用（调用工具 → 收到结果 → 再调用工具 → …），如何判断它已经完成所有工具调用、给出了最终回答？

**判断方式**：

| 方法 | 说明 |
|------|------|
| 检查 `AIMessage` 是否包含 `tool_calls` | 如果最后一条 AI 消息**没有** tool_calls，说明模型认为不需要再调工具了 |
| 使用 `agent.stream()` 流式监听 | 实时观察每一步：tool_call 发起 → 工具执行 → 最终回复 |
| 检查消息列表末尾 | `response["messages"][-1]` 如果是 AIMessage 且 `.content` 非空，通常就是最终回答 |

**演示：用 stream 实时观察 Agent 的每一步决策**

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from rich import print as rprint

# 复用 2.3 节中定义的工具（get_weather、get_exchange_rate、calculate）
# 如果当前 kernel 未执行 2.3 节，请先执行 2.3 节的工具定义代码

print("【实时观察 Agent 的每一步】")
print("-" * 50)

for chunk in agent.stream({
    "messages": [
        HumanMessage(content="查一下东京的天气，然后把 1000 美元换成日元")
    ]
}):
    # chunk 是一个 {node_name: {messages: [...]}} 的字典
    for node_name, node_output in chunk.items():
        messages = node_output.get("messages", [])
        for msg in messages:
            if isinstance(msg, AIMessage):
                if msg.tool_calls:
                    # 模型决定调用工具
                    for tc in msg.tool_calls:
                        print(f"[use tool] {tc['name']}({tc['args']})")
                elif msg.content:
                    # 无 tool_calls → 最终回答
                    print(f"\n[final answer]\n{msg.content}")
            elif isinstance(msg, ToolMessage):
                # 工具返回结果
                preview = str(msg.content)[:100]
                print(f"[tool returned] {preview}...")


In [ ]:
# 案例 1：检查消息类型判断 Agent 是否完成
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage
from rich import print as rprint

# 复用 2.3 节创建的 agent
resp = agent.invoke({
    "messages": [HumanMessage(content="北京天气怎么样？")]
})

print("【逐条分析消息列表】")
for i, msg in enumerate(resp["messages"]):
    msg_type = type(msg).__name__
    if isinstance(msg, AIMessage):
        has_tc = bool(msg.tool_calls)
        has_content = bool(msg.content)
        status = "🔧 调用工具中..." if has_tc else ("✅ 最终回答" if has_content else "⏳ 等待中")
        print(f"  [{i}] {msg_type} | tool_calls={has_tc} | content_len={len(msg.content) if msg.content else 0} | {status}")
    elif isinstance(msg, ToolMessage):
        print(f"  [{i}] {msg_type} | name={msg.name} | 工具已返回结果")
    elif isinstance(msg, HumanMessage):
        print(f"  [{i}] {msg_type} | 用户输入")
    else:
        print(f"  [{i}] {msg_type}")


# 案例 2：编程方式判断 Agent 是否真正完成
def is_agent_finished(response: dict) -> tuple[bool, str]:
    """
    判断 Agent 是否已完成所有工具调用并给出最终回答。
    返回 (是否完成, 原因说明)。
    """
    messages = response.get("messages", [])
    if not messages:
        return False, "消息列表为空"

    last_msg = messages[-1]

    # 检查最后一条消息：必须是 AIMessage，无 tool_calls，有实际内容
    if isinstance(last_msg, AIMessage):
        if last_msg.tool_calls:
            return False, f"最后一条消息仍有 tool_calls: {[tc['name'] for tc in last_msg.tool_calls]}"
        if not last_msg.content:
            return False, "最后一条 AIMessage 内容为空"
        return True, "Agent 已完成并给出最终回答"

    # 如果最后一条是 ToolMessage，说明工具执行完了但模型还没生成最终回答（异常状态）
    if isinstance(last_msg, ToolMessage):
        return False, f"最后一条是工具返回（{last_msg.name}），模型尚未生成最终回答"

    return False, f"最后一条是 {type(last_msg).__name__}，非预期类型"


finished, reason = is_agent_finished(resp)
print(f"\n【完成判定】{reason}")
print(f"  最终回答: {resp['messages'][-1].content[:100] if finished else 'N/A'}...")


# 案例 3：兜底截断 —— 防止模型在最终回答中仍带 tool_calls
def safe_invoke(agent, input_msg: str, max_retries: int = 3) -> str:
    """
    安全调用 Agent：自动检测并处理异常状态。
    - 未完成 → 重试
    - 超限 → 强制要求给出回答
    """
    for attempt in range(max_retries):
        resp = agent.invoke({
            "messages": [HumanMessage(content=input_msg)]
        })
        finished, reason = is_agent_finished(resp)

        if finished:
            return resp["messages"][-1].content

        print(f"[重试 {attempt + 1}/{max_retries}] {reason}")
        # 追问模型，要求给出最终回答
        input_msg = "请基于已有信息直接给出最终回答，不要再调用工具。"

    # 最终兜底：返回最后一条有内容的消息
    for msg in reversed(resp["messages"]):
        if isinstance(msg, AIMessage) and msg.content:
            return f"[兜底截断] {msg.content}"
    return "[错误] 未能获取有效回答"

result = safe_invoke(agent, "查一下北京天气")
print(f"\n【安全调用结果】\n{result[:200]}...")

## 6. How to Limit Tool Calls


**问题**：Agent 有时会陷入"调用工具 -> 不满意 -> 再调用 -> 再调用"的循环，或对简单问题反复调用工具，消耗大量 token 和时间。

下面把每种方式拆成独立代码 cell。先运行公共准备 cell，再分别运行每个方式的示例。每个示例都会打印工具调用轨迹和拦截统计。

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware, before_model
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from rich import print as rprint

load_dotenv(".env")

DEMO_TOOLS = [get_weather, get_exchange_rate, calculate]
BLOCK_PREFIX = "Tool call limit exceeded"
MAX_TOOL_CALLS = 4
RECURSION_LIMIT = MAX_TOOL_CALLS * 3 + 5

limit_demo_model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)


def tool_messages(messages) -> list[ToolMessage]:
    """取出所有 ToolMessage，包括真实工具结果和被限流拦截的结果。"""
    return [m for m in messages if isinstance(m, ToolMessage)]


def count_successful_tool_results(messages) -> int:
    """统计真正执行成功的工具结果数。"""
    return sum(
        not str(m.content).startswith(BLOCK_PREFIX)
        for m in tool_messages(messages)
    )


def count_blocked_tool_results(messages) -> int:
    """统计被 ToolCallLimitMiddleware 拦截的工具调用数。"""
    return sum(
        str(m.content).startswith(BLOCK_PREFIX)
        for m in tool_messages(messages)
    )


def print_tool_trace(response: dict, *, max_success: int | None = None) -> None:
    """打印模型请求了哪些工具，以及哪些工具真正执行或被拦截。"""
    messages = response["messages"]
    print("\n[工具轨迹]")
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"  请求工具 -> {tool_call['name']}({tool_call['args']})")
        elif isinstance(msg, ToolMessage):
            blocked = str(msg.content).startswith(BLOCK_PREFIX)
            status = "已拦截" if blocked else "已执行"
            preview = str(msg.content).replace("\n", " ")[:90]
            print(f"  工具结果 <- {msg.name}: {status} | {preview}")

    success = count_successful_tool_results(messages)
    blocked = count_blocked_tool_results(messages)
    print("\n[统计]")
    if max_success is None:
        print(f"  成功执行：{success}")
    else:
        print(f"  成功执行：{success}/{max_success}")
    print(f"  被拦截：{blocked}")


def print_final_answer(response: dict, limit: int = 900) -> None:
    """打印最终回答，避免调试输出淹没正文。"""
    print("\n[最终回答]")
    rprint(response["messages"][-1].content[:limit])

#### Method 1: recursion_limit (Hard Limit)

用于限制 LangGraph 图节点执行次数。它是最后一道硬兜底，触发后会中断整次运行。

In [ ]:
print("【方式 1：recursion_limit 图执行兜底】")

agent_recursion_limited = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt="你是一个助手。需要信息时可以调用工具，工具返回后再回答。",
)

try:
    agent_recursion_limited.invoke(
        {"messages": [HumanMessage(content="查北京天气，然后给出最终回答")]},
        config={"recursion_limit": 2},
    )
except GraphRecursionError as e:
    print("\n[拦截生效]")
    print("  recursion_limit=2 太低，任务至少需要 model -> tools -> model。")
    print(f"  LangGraph 已终止执行：{type(e).__name__}")

#### Method 2: System Prompt Guidance

通过规则告诉模型不要重复调用相同参数的工具。这种方式成本最低，但依赖模型遵守指令。

In [ ]:
print("【方式 2：System Prompt 引导少调用】")

system_prompt="""你是一个助手，可以使用工具查询信息。
规则：
1. 入参相同时，只调用一次工具
2. 不要追问确认，直接回答
"""
agent_prompt_limited = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt=(
        "你是一个助手，可以使用工具查询信息。\n"
        "规则：\n"
        "1. 入参相同时，只调用一次工具\n"
        "2. 不要追问确认，直接回答。"
    ),
)

response = agent_prompt_limited.invoke(
    {"messages": [HumanMessage(content="帮我查询10次北京的天气，每次查询的结果都给我")]},
    config={"recursion_limit": RECURSION_LIMIT},
)
rprint(response)
print_tool_trace(response)
print_final_answer(response)

#### Method 3: ToolCallLimitMiddleware

在工具执行层限制成功调用次数。即使模型一次性请求 10 次工具，也只允许前 N 次真正执行。

In [ ]:
print("【方式 3：ToolCallLimitMiddleware 执行层拦截】")

agent_tool_limited = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt=(
        "你是一个工具限流演示助手。\n"
        "如果用户要求查询 N 次，请发起 N 次 get_weather 工具调用。\n"
        "如果工具返回限流错误，请在最终回答里标出哪些次数被拦截。"
    ),
    middleware=[ToolCallLimitMiddleware(run_limit=MAX_TOOL_CALLS, exit_behavior="continue")],
)

response = agent_tool_limited.invoke(
    {"messages": [HumanMessage(content="请调用 get_weather 查询北京天气 10 次，并逐次列出结果")]},
    config={"recursion_limit": RECURSION_LIMIT},
)

print_tool_trace(response, max_success=MAX_TOOL_CALLS)
print_final_answer(response)

#### Best Practice: Three-Layer Defense

用 prompt 减少不必要调用，用 middleware 做工具层限流，用 `recursion_limit` 防止图执行失控。

In [ ]:
print("【最佳实践：Prompt 引导 + Middleware 限流 + recursion_limit 兜底】")

BEST_MAX_TOOL_CALLS = 6
BEST_WARN_REMAINING = 2
BEST_RECURSION_LIMIT = BEST_MAX_TOOL_CALLS * 3 + 5


def has_limit_notice(messages, prefix: str) -> bool:
    """避免同一种限制提醒被重复注入到上下文里。"""
    return any(
        isinstance(m, HumanMessage)
        and isinstance(m.content, str)
        and m.content.startswith(prefix)
        for m in messages
    )


@before_model
def gentle_limit(state, runtime):
    """接近工具上限时提醒模型收束。真正的硬拦截交给 ToolCallLimitMiddleware。"""
    messages = state.get("messages", [])
    success_count = count_successful_tool_results(messages)
    remaining = BEST_MAX_TOOL_CALLS - success_count

    if remaining <= 0 and not has_limit_notice(messages, "[工具调用限制]"):
        return {"messages": messages + [HumanMessage(content=(
            f"[工具调用限制] 已成功执行 {success_count} 次工具，达到上限 {BEST_MAX_TOOL_CALLS}。"
            "必须基于已有结果回答，不要再调用工具。"
        ))]}

    if remaining == BEST_WARN_REMAINING and not has_limit_notice(messages, "[工具调用提醒]"):
        return {"messages": messages + [HumanMessage(content=(
            f"[工具调用提醒] 还可成功调用 {remaining} 次工具。请尽快收束回答。"
        ))]}

    return None


agent_best_practice = create_agent(
    model=limit_demo_model,
    tools=DEMO_TOOLS,
    system_prompt=(
        "你是一个智能助手。\n"
        "规则：\n"
        "1. 每个子任务最多调用 1 次工具。\n"
        "2. 相同参数的工具调用不要重复执行。\n"
        "3. 如果用户要求重复查询同一参数，只调用 1 次工具，并把这一次结果重复列出用户要求的次数。\n"
        "4. 工具返回结果后必须直接整合回答，不要追问确认。"
    ),
    middleware=[
        gentle_limit,
        ToolCallLimitMiddleware(run_limit=BEST_MAX_TOOL_CALLS, exit_behavior="continue"),
    ],
    checkpointer=InMemorySaver(),
)

response = agent_best_practice.invoke(
    {"messages": [HumanMessage(content="反复查询北京天气 10 次，每次都告诉我结果")]},
    config={
        "recursion_limit": BEST_RECURSION_LIMIT,
        "configurable": {"thread_id": "tool-limit-best-practice-demo"},
    },
)

print_tool_trace(response, max_success=BEST_MAX_TOOL_CALLS)
print_final_answer(response)